# Project 3: Product Catalogue Checker

**The scenario:** Your company receives a product catalogue from a supplier — 50 products with names, categories, prices, and stock levels. Before this data can be loaded into the company database, someone has to check it. Right now that person is you, and you do it manually in Excel.

Today you write a script that does it automatically.

**What this connects to:** In future modules you will ingest data from APIs, databases, and web scrapers. Every one of those sources sends dirty data. The validation logic you write today is the same logic you will use there — just applied to different inputs.

**What you will build:**
1. A set of validation rules defined as functions
2. A loop that applies every rule to every product
3. Three output files: valid products, invalid products, products that need human review

**Concepts practised:** functions, dictionaries, lists, for loops, if/elif/else, sets, file writing, try/except

---

## Step 1 — Load and inspect the catalogue

Nothing new here — you have done this before. Run the cell and study the output carefully.
Before writing any validation logic, you need to understand what problems exist in the data.

In [1]:
import pandas as pd

df = pd.read_csv('products_raw.csv')

In [2]:
df.head()

,product_id,name,category,price,stock,supplier
0,P001,Wireless Mouse,Electronics,29.99,150,TechSupply Co
1,P002,USB-C Hub,Electronics,49.99,83,TechSupply Co
2,P003,Standing Desk,Furnitre,299.99,12,OfficePro
3,P004,Ergonomic Chair,Furniture,449.99,8,OfficePro
4,P005,Laptop Stand,Electronics,39.99,-5,TechSupply Co


In [3]:
list(df['category'].value_counts().index)

['Electronics',
 'Stationery',
 'Health',
 'Appliances',
 'Kitchenware',
 'Furniture',
 'Decoration',
 'Furnitre',
 'Lighting',
 'Electronis',
 'Accessories']

In [4]:
missing_price

NameError: name 'missing_price' is not defined

In [5]:
# What categories exist in the catalogue?

all_categories = df['category'].drop_duplicates().tolist()
print('All categories found:')
for cat in sorted(all_categories):
    print(' -', cat)

print('-' * 40)

# How many products are missing a price?
missing_price = df[df['price'].isna() | (df['price'].astype(str).str.strip() == '')]

print(f'Products with missing price: {len(missing_price)}')

for _, row in missing_price.iterrows():
    print(f" {row['product_id']} - {row['name']}")

All categories found:
 - Accessories
 - Appliances
 - Decoration
 - Electronics
 - Electronis
 - Furnitre
 - Furniture
 - Health
 - Kitchenware
 - Lighting
 - Stationery
----------------------------------------
Products with missing price: 4
 P007 - Webcam HD
 P017 - Smart Bulb Pack
 P036 - Shredder
 P044 - Water Dispenser


You should see some category names that are clearly typos — `Furnitre`, `Electronis`.  
You should also see four products with no price at all.

This is exactly what data ingestion looks like before you build automated checks.

---

## Step 2 — Define the valid categories and business rules

Before writing validation functions, we need to decide what "valid" means.  
In a real ETL pipeline, these rules come from the business or from a database schema.

Here are our rules:
- **Price** must be present and greater than zero
- **Stock** must be zero or greater (negative stock is impossible)
- **Category** must be one of the approved values (no typos allowed)
- **Product ID** must start with 'P' and be followed by exactly 3 digits

A product can be:
- `valid` — passes all rules, ready to load
- `invalid` — fails a hard rule (missing price, negative stock)
- `needs_review` — something looks wrong but could be fixed (category typo)

In [6]:
[x for _, x in df.iterrows()]

[product_id              P001
 name          Wireless Mouse
 category         Electronics
 price                  29.99
 stock                    150
 supplier       TechSupply Co
 Name: 0, dtype: object,
 product_id             P002
 name              USB-C Hub
 category        Electronics
 price                 49.99
 stock                    83
 supplier      TechSupply Co
 Name: 1, dtype: object,
 product_id             P003
 name          Standing Desk
 category           Furnitre
 price                299.99
 stock                    12
 supplier          OfficePro
 Name: 2, dtype: object,
 product_id               P004
 name          Ergonomic Chair
 category            Furniture
 price                  449.99
 stock                       8
 supplier            OfficePro
 Name: 3, dtype: object,
 product_id             P005
 name           Laptop Stand
 category        Electronics
 price                 39.99
 stock                    -5
 supplier      TechSupply Co
 Name: 4, dt

In [7]:
# Approved categories — anything outside this set is flagged
VALID_CATEGORIES = {
    'Electronics', 'Furniture', 'Stationery', 'Lighting',
    'Health', 'Decoration', 'Accessories', 'Appliances',
    'Kitchenware'
}

# Minimum and maximum acceptable price
MIN_PRICE = 0.01
MAX_PRICE = 10000.00

print('Validation rules set.')
print(f'Valid categories: {sorted(VALID_CATEGORIES)}')
print(f'Price range: €{MIN_PRICE} — €{MAX_PRICE}')

Validation rules set.
Valid categories: ['Accessories', 'Appliances', 'Decoration', 'Electronics', 'Furniture', 'Health', 'Kitchenware', 'Lighting', 'Stationery']
Price range: €0.01 — €10000.0


---
## Step 3 — Write the validation functions

Each rule becomes its own function. This is a key design principle:  
**one function, one responsibility.**

When a new rule is added later, you add one new function — you don't rewrite everything.
This is exactly how validation is structured in real data pipelines.

Each function takes a product dictionary and returns a tuple: `(passed, message)`  
- `passed` is `True` or `False`  
- `message` explains what went wrong (empty string if nothing did)

In [16]:
def validate_price(product):
    """
    Checks that price is present and within the valid range.
    Returns (True, '') if valid, or (False, reason) if not.
    """        
    raw = product['price']

    if not raw:
        return False, 'missing price'

    if isinstance(raw, str):
        return False, 'price is a string'

    try:
        price = float(raw)
        
    except ValueError:
        return False, f'price is not a number: {raw!r}'

    if price < MIN_PRICE:
        return False, f'price too low: {price}'
    if price > MAX_PRICE:
        return False, f'price too high: {price}'

    return True, ''
    


def validate_stock(product):
    """
    Checks that stock is a non-negative integer.
    """
    raw = product['stock']

    if not raw:
        return False, 'missing stock'
    
    try:
        stock = int(raw)
    
    except ValueError:
        return False, f'stock is not a number: {raw!r}'

    if stock < 0:
        return False, f'negative stock: {stock}'

    return True, ''


# Test both functions on a couple of example products
test_cases = [
    {'product_id': 'P007', 'name': 'Webcam HD',  'price': '',    'stock': '27', 'category': 'Electronics'},
    {'product_id': 'P005', 'name': 'Laptop Stand','price': '39.99','stock': '-5','category': 'Electronics'},
    {'product_id': 'P001', 'name': 'Wireless Mouse','price':'29.99','stock': '150','category': 'Electronics'},
]

print('Testing validate_price():')
for p in test_cases:
    ok, msg = validate_price(p)
    print(f"  {p['product_id']} — passed: {ok}  {msg}")

print()
print('Testing validate_stock():')
for p in test_cases:
    ok, msg = validate_stock(p)
    print(f"  {p['product_id']} — passed: {ok}  {msg}")

Testing validate_price():
  P007 — passed: False  missing price
  P005 — passed: False  price is a string
  P001 — passed: False  price is a string

Testing validate_stock():
  P007 — passed: True  
  P005 — passed: False  negative stock: -5
  P001 — passed: True  


Now write the remaining two validation functions yourself.

In [9]:
def validate_category(product):
    """
    Checks that the category is in the approved set.
    Returns (True, '') if valid.
    Returns (False, reason) if the category is missing or not in VALID_CATEGORIES.
    """
    # YOUR CODE HERE
    # Hint: use 'in' to check set membership — e.g. 'Electronics' in VALID_CATEGORIES
    # Hint: use product['category'].strip() to get the category value

    if product['category'].strip() in VALID_CATEGORIES:
        return True, ''
    else:
        return False, f"invalid category: {product['category']}"

def validate_product_id(product):
    """
    Checks that product_id starts with 'P' and the rest are digits.
    Example of a valid ID: 'P001', 'P042'
    Example of an invalid ID: 'X001', 'P99', 'ABC'
    """
    # YOUR CODE HERE
    # Hint: pid = product['product_id'].strip()
    # Hint: pid.startswith('P') checks the first character
    # Hint: pid[1:].isdigit() checks that everything after 'P' is a digit
    # Hint: len(pid[1:]) == 3 checks there are exactly 3 digits
    pid = product['product_id'].strip()
    
    if not pid.startswith('P'):
        return False, f"product_id does not start with 'P': {product['product_id']}"

    
    if not pid[1:].isdigit():
        return False, f"product_id has invalid format: {product['product_id']}"

    if len(pid[1:]) != 3:
        return False, f"product_id has wrong length: {product['product_id']}"

    return True, ''


# Test your functions:
print('Testing validate_category():')
cat_tests = [
    {'product_id': 'P003', 'category': 'Furnitre'},    # typo — should fail
    {'product_id': 'P004', 'category': 'Furniture'},   # correct
    {'product_id': 'P012', 'category': 'Electronis'},  # typo — should fail
]

for p in cat_tests:
    ok, msg = validate_category(p)
    print(f"  {p['product_id']} '{p['category']}' — passed: {ok}  {msg}")

print()
print('Testing validate_product_id():')
id_tests = [
    {'product_id': 'P001'},  # valid
    {'product_id': 'P99'},   # too short
    {'product_id': 'X001'},  # wrong prefix
    {'product_id': 'P042'},  # valid
]
for p in id_tests:
    ok, msg = validate_product_id(p)
    print(f"  '{p['product_id']}' — passed: {ok}  {msg}")

Testing validate_category():
  P003 'Furnitre' — passed: False  invalid category: Furnitre
  P004 'Furniture' — passed: True  
  P012 'Electronis' — passed: False  invalid category: Electronis

Testing validate_product_id():
  'P001' — passed: True  
  'P99' — passed: False  product_id has wrong length: P99
  'X001' — passed: False  product_id does not start with 'P': X001
  'P042' — passed: True  


---
## Step 4 — Run all rules against all products

Now we combine everything. For each product, we run all four validation functions  
and collect the results.

**Key design decision:** we collect ALL failures for each product — not just the first one.  
This is how production validators work: show the user everything wrong at once,  
not one error at a time.

**Status logic:**
- Any `invalid` rule failure → status is `invalid`
- Any `needs_review` failure (category typo) → status is `needs_review`  
- All rules pass → status is `valid`

In [19]:
# Group rules by severity
# 'invalid' rules: hard failures — data cannot be loaded
invalid_rules = [validate_price, validate_stock, validate_product_id]

# 'needs_review' rules: soft failures — human should check
review_rules = [validate_category]

# Output buckets
valid_products   = []
invalid_products = []
review_products  = []

for _, product in df.iterrows():
    failures = []     # list of failure messages for this product
    status = 'valid'  # assume valid until a rule fails

    # Check hard rules first
    for rule in invalid_rules:
        passed, message = rule(product)
        if not passed:
            failures.append(message)
            status = 'invalid'

    # Check soft rules (only relevant if not already invalid)
    for rule in review_rules:
        passed, message = rule(product)
        if not passed:
            failures.append(message)
            if status == 'valid':   # don't downgrade from 'invalid'
                status = 'needs_review'

    # Attach validation results to the product
    product['status']   = status
    product['issues']   = '; '.join(failures) if failures else ''

    # Route to the right bucket
    if status == 'valid':
        valid_products.append(product)
    elif status == 'invalid':
        invalid_products.append(product)
    else:
        review_products.append(product)

print(f'Results:')
print(f'  Valid:        {len(valid_products)}')
print(f'  Needs review: {len(review_products)}')
print(f'  Invalid:      {len(invalid_products)}')
print(f'  Total:        {len(df)}')
print()
print('Invalid products and their issues:')
for p in invalid_products:
    print(f"  {p['product_id']} {p['name']}: {p['issues']}")
print()


print('Products needing review:')
for p in review_products:
    print(f"  {p['product_id']} {p['name']}: {p['issues']}")

Results:
  Valid:        42
  Needs review: 3
  Invalid:      5
  Total:        50

Invalid products and their issues:
  P005 Laptop Stand: negative stock: -5
  P014 Noise Cancelling Headphones: negative stock: -2
  P024 Air Purifier: negative stock: -1
  P033 Monitor Riser: negative stock: -3; invalid category: Furnitre
  P050 Dish Rack: negative stock: -4

Products needing review:
  P003 Standing Desk: invalid category: Furnitre
  P012 Cable Management Kit: invalid category: Electronis
  P019 Conference Microphone: invalid category: Electronis


**What just happened — and why it matters for ETL:**

You just built a validation layer. In a real data pipeline this sits between ingestion (reading the raw data) and loading (writing it to the database). Nothing reaches the database unless it passes validation.

The `issues` column is called an **audit trail** — it tells you exactly why each row was rejected, so you can fix the source and re-run.

---

## Step 5 — Export three files

The output is three separate CSVs: one for each status.  
We write a reusable export function — the same pattern from Project 2.

**Your turn:** the export function below is missing its body. Write it.

,product_id,name,category,price,stock,supplier,status,issues
0,P001,Wireless Mouse,Electronics,29.99,150,TechSupply Co,valid,
1,P002,USB-C Hub,Electronics,49.99,83,TechSupply Co,valid,
3,P004,Ergonomic Chair,Furniture,449.99,8,OfficePro,valid,
5,P006,Monitor 27inch,Electronics,379.99,34,TechSupply Co,valid,
6,P007,Webcam HD,Electronics,NaN,27,TechSupply Co,valid,
7,P008,Desk Lamp,Lighting,24.99,200,BrightSpace,valid,
8,P009,Whiteboard,Stationery,89.99,15,OfficeEssentials,valid,
9,P010,Sticky Notes Pack,Stationery,4.99,500,OfficeEssentials,valid,
10,P011,Mechanical Keyboard,Electronics,129.99,41,TechSupply Co,valid,
12,P013,Office Plant,Decoration,19.99,60,GreenOffice,valid,


In [ ]:
vp = pd.DataFrame(valid_products)
rp = pd.DataFrame(review_products)
ip = pd.DataFrame(invalid_products)

vp.to_csv('products_valid.csv', index=False)
rp.to_csv('products_needs_review.csv', index=False)
ip.to_csv('products_invalid.csv', index=False)  

Open each file in Excel. You should see:
- `products_valid.csv` — clean products, ready to load
- `products_needs_review.csv` — category typos for a human to fix
- `products_invalid.csv` — hard failures, each with an `issues` column explaining why

---

## Step 6 — Summary report and reflection

Finally, print a summary that a manager could read — no technical jargon, just numbers.

In [ ]:
total = len(products)
valid_pct   = len(valid_products)   / total * 100
review_pct  = len(review_products)  / total * 100
invalid_pct = len(invalid_products) / total * 100

print('=' * 45)
print('  CATALOGUE VALIDATION REPORT')
print('=' * 45)
print(f'  Total products checked:  {total}')
print(f'  Ready to load:           {len(valid_products):3d}  ({valid_pct:.0f}%)')
print(f'  Need human review:       {len(review_products):3d}  ({review_pct:.0f}%)')
print(f'  Rejected (invalid):      {len(invalid_products):3d}  ({invalid_pct:.0f}%)')
print('=' * 45)

# Most common issue type
issue_counts = {}
for p in invalid_products + review_products:
    for issue in p['issues'].split('; '):
        issue_type = issue.split(':')[0]   # take just the label, not the value
        issue_counts[issue_type] = issue_counts.get(issue_type, 0) + 1

print()
print('Most common issues:')
for issue, count in sorted(issue_counts.items(), key=lambda x: x[1], reverse=True):
    print(f'  {issue}: {count}')

**Reflect:**
1. What would happen if the supplier sent next month's catalogue with the same typos? How long would validation take?
2. How would you add a new rule — for example, flagging any product with a price over €5000?
3. In a real ETL pipeline, what would happen after validation? Where would the valid products go?

**Extension:** Count how many products per supplier are invalid. Which supplier sends the most problems?

In [ ]:
# Extension: invalid products by supplier
# YOUR CODE HERE
# Hint: build a dictionary — key = supplier name, value = count of invalid products
